In [72]:
from pymatgen.core.structure import Structure
from ase.geometry.geometry import get_layers
from vdW_structures.vdW_structure import VdWStructure
import numpy as np
from typing import List, Tuple

In [99]:
s = Structure.from_file('original_structures/BiSe2_slab.vasp')
#s = Structure.from_file('original_structures/Bi2Se3.vasp')
vdW_s = VdWStructure(s, minimum_vdW_gap=2.4)
vdW_s = vdW_s.z_solve_shift_vdW_layers(layer_inds=[0], solve_shift=[4], adjust_z_cell=True)
vdW_s.vdW_spacings

[4.053074184176202, 2.935904452246059]

In [100]:
def get_vdW_layers(structure: Structure, minimum_vdW_gap=2.4):
    """Identify van der Waals layers in the structure based on spacing along the (0, 0, 1) Miller index."""
    
    vdW_layers, z_images, vdW_spacings = [[]], [[]], []
    atoms = structure.to_ase_atoms()
    layers, layer_dist = get_layers(atoms, (0, 0, 1))

    #print(layers, layer_dist)
    
    # Convert layers into the [[atomic layers], spacings]
    atomic_layers, atomic_spacings = [[] for i in range(max(layers)+1)], [None for i in range(max(layers)+1)]
    for site_ind, layer_ind in enumerate(layers):
        atomic_layers[layer_ind].append(site_ind)
        atomic_spacings[layer_ind] = layer_dist[layer_ind]
        
    #print(atomic_layers, atomic_spacings)
    
    # Get z-coordinates sorted by layer distance
    site_values = [layer_dist[layers[i]] for i in range(len(atoms))]
    sorted_indices = sorted(range(len(atoms)), key=lambda i: site_values[i])
    sorted_zs = [site_values[i] for i in sorted_indices]

    # Compute layer spacings, accounting for periodic boundaries
    diffs = [structure.lattice.c - sorted_zs[-1] + sorted_zs[0]] + [
        sorted_zs[i + 1] - sorted_zs[i] for i in range(len(sorted_zs) - 1)
    ]

    # Group atoms into vdW layers
    layer_index = 0
    for i, diff in enumerate(diffs):
        if diff > minimum_vdW_gap:
            vdW_spacings.append(diff)
            vdW_layers.append([])
            z_images.append([])
            layer_index += 1
        vdW_layers[layer_index].append(sorted_indices[i])
        z_images[layer_index].append(0)

    # Handle periodic boundary conditions
    if len(vdW_layers) > 1 and diffs[0] < minimum_vdW_gap:
        z_images[0] = [-1] * len(vdW_layers[0]) + [0] * len(vdW_layers[-1])
        vdW_layers[0] += vdW_layers.pop()
        z_images.pop()

    return [layer for layer in vdW_layers if layer], [z_images[i] for i, layer in enumerate(vdW_layers) if layer], vdW_spacings, atomic_layers, atomic_spacings


def shift_to_vdW_gap(
    structure: Structure, 
    layers: List[List[int]], 
    images: List[List[int]], 
    spacings: List[float], 
    atomic_layers: List[List[float]], 
    atomic_spacings: List[float]
):

    def find_sublist_index(lists: list[list[int]], target: int) -> int:
        for i, sublist in enumerate(lists):
            if target in sublist:
                return i  # Return the index as soon as we find the target
        return -1

    # Identify the largest vdW gap and its index
    max_gap_idx = spacings.index(max(spacings))

    c_vector = structure.lattice.matrix[2]

    # Identify the largest site index from the below gap and 
    if not max_gap_idx == 0: # Ignore if the largest gap is already across the PBC
        top_site_ind = layers[max_gap_idx][0]
        bottom_site_ind = layers[max_gap_idx - 1][-1]
        top_distance = atomic_spacings[find_sublist_index(atomic_layers, top_site_ind)]
        bottom_distance = atomic_spacings[find_sublist_index(atomic_layers, bottom_site_ind)]
        shift_distance = -1 * np.average([top_distance, bottom_distance])
        shift_vector = shift_distance * (c_vector / np.linalg.norm(c_vector))
        print(shift_vector)

        for site in structure:
            site.coords += shift_vector
            
        return Structure(
            lattice=structure.lattice,
            coords=[site.coords for site in structure],
            species=[site.specie for site in structure],
            coords_are_cartesian=True,
            to_unit_cell=True
            ).get_sorted_structure()

    else:
        return structure


In [101]:
layers, images, spacings, atomic_layers, atomic_spacings = get_vdW_layers(vdW_s.structure)
print(layers, images, spacings, atomic_layers, atomic_spacings)
'''
shifted_s = shift_to_vdW_gap(
    structure= vdW_s.structure, 
    layers= layers, 
    images= images, 
    spacings= spacings,
    atomic_layers=atomic_layers,
    atomic_spacings=atomic_spacings
)
s_layers, s_images, s_spacings, s_atomic_layers, s_atomic_spacings = get_vdW_layers(shifted_s)
print(s_layers, s_images, s_spacings)
'''

[[8, 1, 4, 5, 0, 9], [10, 3, 6, 7, 2, 11]] [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]] [4.053074184176202, 2.935904452246059] [[8], [1], [4], [5], [0], [9], [10], [3], [6], [7], [2], [11]] [2.290107083779596, 2.994003526732217, 3.2876766256722334, 4.437926432997699, 4.731599531937719, 5.435495974890334, 8.371400427136393, 8.960167885072961, 9.429028389346314, 10.371894248570655, 10.840754752844, 11.429522210780569]


'\nshifted_s = shift_to_vdW_gap(\n    structure= vdW_s.structure, \n    layers= layers, \n    images= images, \n    spacings= spacings,\n    atomic_layers=atomic_layers,\n    atomic_spacings=atomic_spacings\n)\ns_layers, s_images, s_spacings, s_atomic_layers, s_atomic_spacings = get_vdW_layers(shifted_s)\nprint(s_layers, s_images, s_spacings)\n'

In [102]:
shifted_s = shift_to_vdW_gap(
    structure= vdW_s.structure, 
    layers= layers, 
    images= images, 
    spacings= spacings,
    atomic_layers=atomic_layers,
    atomic_spacings=atomic_spacings
)
s_layers, s_images, s_spacings, s_atomic_layers, s_atomic_spacings = get_vdW_layers(shifted_s)
print(s_layers, s_images, s_spacings, s_atomic_layers, s_atomic_spacings)

[[8, 1, 4, 5, 0, 9], [10, 3, 6, 7, 2, 11]] [[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]] [4.053074184176202, 2.935904452246059] [[8], [1], [4], [5], [0], [9], [10], [3], [6], [7], [2], [11]] [2.290107083779596, 2.994003526732217, 3.2876766256722334, 4.437926432997699, 4.731599531937719, 5.435495974890334, 8.371400427136393, 8.960167885072961, 9.429028389346314, 10.371894248570655, 10.840754752844, 11.429522210780569]


In [97]:
vdW_s.structure.lattice.matrix

array([[ 2.09568326, -3.62982989,  0.        ],
       [ 2.09568326,  3.62982989,  0.        ],
       [ 0.        ,  0.        , 30.99278275]])